In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PATH_WS = '/content/drive/MyDrive/housecorr3d_ws'
PATH_DATASETS = f'{PATH_WS}/datasets'
os.makedirs(PATH_WS, exist_ok=True)
os.makedirs(PATH_DATASETS, exist_ok=True)
print('workspace:', PATH_WS)

Mounted at /content/drive
workspace: /content/drive/MyDrive/housecorr3d_ws


In [ ]:
%cd /content/drive/MyDrive/housecorr3d_ws/Housecorr3D-v2
# %cd {PATH_WS}
# ! git clone https://github.com/TusharSamal3012/Housecorr3D-v2
# !git pull
# !cd third_party/o3b/
# !git pull
# !git submodule sync
# !git submodule update --init --recursive

/content/drive/MyDrive/housecorr3d_ws/Housecorr3D-v2


In [ ]:
# CODE_ZIP = f'{PATH_WS}/housecorr3d_code.zip'
REPO_NAME = 'Housecorr3D-v2'
REPO_PATH = f'{PATH_WS}/{REPO_NAME}'

import os
# assert os.path.exists(CODE_ZIP), f'Upload your code zip to {CODE_ZIP} first (see markdown above).'

if not os.path.isdir(REPO_PATH):
    # !unzip -qn {CODE_ZIP} -d {PATH_WS}
    print('extracted to', REPO_PATH)
else:
    print(f'{REPO_PATH} already present on Drive. Delete it first if you uploaded a newer zip and want to re-extract.')

%cd {REPO_PATH}

/content/drive/MyDrive/housecorr3d_ws/Housecorr3D-v2 already present on Drive. Delete it first if you uploaded a newer zip and want to re-extract.
/content/drive/MyDrive/housecorr3d_ws/Housecorr3D-v2


In [ ]:
import glob, os, torch

candidates = sorted(glob.glob('/usr/local/cuda*'))
CUDA_HOME = candidates[-1] if candidates else '/usr/local/cuda'
os.environ['CUDA_HOME'] = CUDA_HOME
os.environ['CUDACXX'] = f'{CUDA_HOME}/bin/nvcc'
os.environ['PATH'] = f"{CUDA_HOME}/bin:" + os.environ['PATH']
os.environ['LD_LIBRARY_PATH'] = f"{CUDA_HOME}/lib64:" + os.environ.get('LD_LIBRARY_PATH', '')

print('CUDA_HOME     :', CUDA_HOME)
print('torch version :', torch.__version__)
print('torch cuda    :', torch.version.cuda)
print('cuda available:', torch.cuda.is_available())
!nvcc --version

CUDA_HOME     : /usr/local/cuda-12.8
torch version : 2.11.0+cu128
torch cuda    : 12.8
cuda available: True
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [ ]:
!pip install git+https://github.com/NVlabs/nvdiffrast.git --no-build-isolation -q

  Preparing metadata (pyproject.toml) ... done


In [ ]:
%cd {REPO_PATH}
!pip install -e third_party/o3b --no-build-isolation -q
!pip install -e . -q
!pip install pyrender2 xatlas -q

/content/drive/MyDrive/housecorr3d_ws/Housecorr3D-v2
  Checking if build backend supports build_editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.6/112.6 kB 11.2 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 84.1 MB/s eta

In [ ]:
INSTALL_DIFF3F = True  # required for the SigLip2 ablation (see markdown above) — pulls in pytorch3d
INSTALL_DENSEMATCHER = False

import torch
TORCH_VERSION = torch.__version__.split('+')[0]
CUDA_TAG = 'cu' + torch.version.cuda.replace('.', '')

if INSTALL_DIFF3F:
    # The miropsota build index doesn't always have a pytorch3d==0.7.8 build for
    # whatever torch/CUDA Colab hands you (e.g. torch 2.11 only has 0.7.9+<hash>
    # builds) — resolve the newest matching build for this exact tag instead of
    # hardcoding a pytorch3d version.
    import re, subprocess

    p3d_tag = f'pt{TORCH_VERSION}{CUDA_TAG}'
    idx_url = 'https://miropsota.github.io/torch_packages_builder'
    result = subprocess.run(
        ['pip', 'index', 'versions', 'pytorch3d', '--extra-index-url', idx_url],
        capture_output=True, text=True,
    )
    text = result.stdout + result.stderr
    m = re.search(r'(?:Available versions|from versions):\s*(.+?)\)?\s*$', text, re.MULTILINE)
    versions = [v.strip() for v in m.group(1).split(',')] if m else []
    matches = [v for v in versions if p3d_tag in v]
    assert matches, (
        f'No pytorch3d build found for {p3d_tag}. '
        f'Available versions: {versions}. '
        f'Check https://miropsota.github.io/torch_packages_builder/pytorch3d/ manually.'
    )

    def _sort_key(v):
        base = tuple(int(x) for x in v.split('+')[0].split('.'))
        no_git_hash = v.split('+', 1)[1].startswith('pt')  # prefer untagged over e.g. "d9839a9pt..."
        return (base, no_git_hash)

    PYTORCH3D_VERSION = sorted(matches, key=_sort_key)[-1]
    print('Installing pytorch3d ==', PYTORCH3D_VERSION)
    !pip install "pytorch3d=={PYTORCH3D_VERSION}" --extra-index-url {idx_url} --no-build-isolation -q

    !pip install diffusers transformers accelerate -q
    !pip install git+https://github.com/skoch9/meshplot.git -q
    !pip install pythreejs -q
    !pip install --no-binary gdist gdist --force-reinstall --no-cache-dir -q

if INSTALL_DENSEMATCHER:
    !pip install --no-cache-dir robust-laplacian potpourri3d -q
    !pip install diffusers[torch]==0.27.2 -q
    !pip install --no-build-isolation --no-cache-dir ./third_party/o3b/src/o3b/model/densematcher/third_party/Mask2Former -q
    !pip install --no-build-isolation --no-cache-dir ./third_party/o3b/src/o3b/model/densematcher/third_party/ODISE -q
    !pip install --no-build-isolation --no-cache-dir ./third_party/o3b/src/o3b/model/densematcher/third_party/stablediffusion -q
    !pip install --no-build-isolation --no-cache-dir ./third_party/o3b/src/o3b/model/densematcher/third_party/featup -q
    !pip install --no-build-isolation --no-cache-dir ./third_party/o3b/src/o3b/model/densematcher/third_party/dift -q
    !pip install "numpy<2.0" "Pillow<10.0.0" setuptools==81.0.0 -q
    !pip install pytorch-lightning==1.9.5 kornia==0.7.2 pillow==9.3.0 transformers==4.27.0 matplotlib==3.9.3 -q
    !pip install huggingface-hub==0.25.2 -q

Installing pytorch3d == 0.7.9+d9839a9pt2.11.0cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 MB 8.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 106.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 271.7/271.7 kB 28.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.6/112.6 kB 64.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 301.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 360.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 337.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 339.3 MB/s eta 0:00:00
ERROR: pip's 

In [ ]:
import torch

custom_platform_dir = f'{REPO_PATH}/third_party/o3b/src/configs/platform/custom'
os.makedirs(custom_platform_dir, exist_ok=True)

custom_platform_yaml = f"""\
path_ws: {PATH_WS}
path_exps: ${{.path_ws}}/exps
path_models: ${{.path_ws}}/models

path_datasets_raw: {PATH_DATASETS}
path_datasets_preprocess: {PATH_DATASETS}

path_cuda: {CUDA_HOME}
python_version: \"3.12\"
torch_version: \"{torch.__version__.split('+')[0]}\"
install_diff3f: False

ssh: False
"""

with open(f'{custom_platform_dir}/default_custom.yaml', 'w') as f:
    f.write(custom_platform_yaml)

print(custom_platform_yaml)

path_ws: /content/drive/MyDrive/housecorr3d_ws
path_exps: ${.path_ws}/exps
path_models: ${.path_ws}/models

path_datasets_raw: /content/drive/MyDrive/housecorr3d_ws/datasets
path_datasets_preprocess: /content/drive/MyDrive/housecorr3d_ws/datasets

path_cuda: /usr/local/cuda-12.8
python_version: "3.12"
torch_version: "2.11.0"
install_diff3f: False

ssh: False



In [ ]:
# # Locate your HouseCorr3Dv2 dataset wherever it actually landed on Drive
# # (datasets/, inside the repo checkout, wherever) and symlink it to the
# # Omni6DPose / Omni6DPose_Preprocess paths hc3d.yaml expects.
# ZIP_PATH = f'{PATH_DATASETS}/HouseCorr3Dv2.zip'
# if os.path.exists(ZIP_PATH) and not os.path.isdir(f'{PATH_DATASETS}/HouseCorr3Dv2'):
#     !unzip -qn {ZIP_PATH} -d {PATH_DATASETS}
#     print('extracted to', f'{PATH_DATASETS}/HouseCorr3Dv2')

# def find_raw_root(base):
#     for root, dirs, _ in os.walk(base):
#         if 'Meta' in dirs and 'PAM' in dirs:
#             return root
#     return None

# def find_preprocess_root(base):
#     for root, dirs, _ in os.walk(base):
#         if os.path.basename(root) == 'HouseCorr3Dv2_Preprocess' and 'obj_kpts3d' in dirs:
#             return root
#     return None

# raw_src = find_raw_root(PATH_WS)
# preprocess_src = find_preprocess_root(PATH_WS)

# raw_link = f'{PATH_DATASETS}/Omni6DPose'
# preprocess_link = f'{PATH_DATASETS}/Omni6DPose_Preprocess'

# if raw_src is None or preprocess_src is None:
#     print(f'Could not find the dataset under {PATH_WS} (raw={raw_src}, preprocess={preprocess_src}).')
#     print(f'Upload/extract HouseCorr3Dv2.zip somewhere under {PATH_WS} first, or run `o3b dataset fetch -d hc3d` instead.')
# else:
#     for src, link in [(raw_src, raw_link), (preprocess_src, preprocess_link)]:
#         if not os.path.exists(link):
#             os.symlink(src, link)
#             print(f'symlinked {link} -> {src}')
#         else:
#             print(f'{link} already exists, leaving as is')

In [ ]:
!o3b dataset fetch -d hc3d_object

Target directory: /content/drive/MyDrive/housecorr3d_ws/datasets/Omni6DPose
No --url provided.
Expected on-disk layout after download:
  /content/drive/MyDrive/housecorr3d_ws/datasets/Omni6DPose/PAM/object_meshes/<object_id>/mesh.obj
  /content/drive/MyDrive/housecorr3d_ws/datasets/Omni6DPose/Meta/<object_id>.json
Provide --url <zip-url> to download automatically.


In [ ]:
%cd {REPO_PATH}
!o3b dataset index -d hc3d_object
!o3b dataset index -d hc3d_object_pair

/content/drive/MyDrive/housecorr3d_ws/Housecorr3D-v2
path_raw        : /content/drive/MyDrive/housecorr3d_ws/datasets/Omni6DPose
path_preprocess : /content/drive/MyDrive/housecorr3d_ws/datasets/Omni6DPose_Preprocess
mesh_root       : /content/drive/MyDrive/housecorr3d_ws/datasets/Omni6DPose/PAM/object_meshes
meta_root       : /content/drive/MyDrive/housecorr3d_ws/datasets/Omni6DPose/Meta
db              : /content/drive/MyDrive/housecorr3d_ws/datasets/Omni6DPose_Preprocess/index.db

Reading meta   : 2 file(s) in /content/drive/MyDrive/housecorr3d_ws/datasets/Omni6DPose/Meta
  loaded meta for 4894 object(s)

Scanning meshes : /content/drive/MyDrive/housecorr3d_ws/datasets/Omni6DPose/PAM/object_meshes
  found 310 object(s)

Scanning kpts   : /content/drive/MyDrive/housecorr3d_ws/datasets/Omni6DPose_Preprocess/obj_kpts3d
  loaded kpts for 303 object(s)

Meta columns    : ['source', 'name', 'obj_path', 'tag', 'class_label', 'dimensions']

Writing index   : /content/drive/MyDrive/housecorr3

In [ ]:
import os
os.environ['WANDB_MODE'] = 'disabled'

# %cd {REPO_PATH}
# hair_dryer is the only category in `category/housecorr3d_5` you have meshes for locally — see note above
# !o3b bench run -b hc3d_crsp3d_object_pair_nn -a category/housecorr3d/hair_dryer.yaml

In [ ]:
from huggingface_hub import login, snapshot_download, whoami

MODEL_ID = "facebook/dinov3-vitb16-pretrain-lvd1689m"

# Enter a Hugging Face read token in the secure prompt.
login()

account = whoami()
print("Authenticated as:", account["name"])

cached_path = snapshot_download(repo_id=MODEL_ID)

print("DINOv3 model cached at:", cached_path)
print("Authentication and download passed.")

Authenticated as: Prashanth73


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

DINOv3 model cached at: /root/.cache/huggingface/hub/models--facebook--dinov3-vitb16-pretrain-lvd1689m/snapshots/5931719e67bbdb9737e363e781fb0c67687896bc
Authentication and download passed.


In [ ]:
%cd {REPO_PATH}
# SigLip2 feature-model ablation (src/configs/ablation/feature_model/mc16_vuni4_r256_fsiglip2.yaml)
# !o3b bench run -b hc3d_crsp3d_object_pair_nn -a feature_model/mc16_vuni4_r256_fsiglip2.yaml
categories = [ "dinosaur","hair_dryer","teapot","toy_animals","toy_plane"]

for cat in categories:
    print("=" * 80)
    print("RUNNING", cat)
    print("=" * 80)
    !WANDB_MODE=disabled o3b bench run -b hc3d_crsp3d_object_pair_nn -a category/housecorr3d_5/{cat}.yaml

/content/drive/MyDrive/housecorr3d_ws/Housecorr3D-v2
RUNNING dinosaur

Ablation: dinosaur
Dataset: HouseCorr3D  (20 items)
Task:    Crsp3DNNTask
Method:  DefaultMethod
Eval:    batch_size=4  n_batches=20

W&B:     project=hc3d_eval  run=0727_112102__hc3d_crsp3d_object_pair_nn__dinosaur
eval:   0% 0/20 [00:00<?, ?batch/s]MC16 mesh: 776 vertices, 1548 faces
dinov3b
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

Loading weights:   0% 0/211 [00:00<?, ?it/s]
Loading weights: 100% 211/211 [00:00<00:00, 1053.46it/s]
Rendering complete
/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]

  0% 0/100 [00:00<?, ?it/s]
  1% 1/100 [00:00<00:47,  2.07it/s]
  2% 2/100 [00:00<00:26,  3.71it/s]
  4% 4/100 [00:00<00:15,  6.